<a href="https://colab.research.google.com/github/MParvan/ecg-biometrics-bench/blob/main/Custom_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Injecting Custom PyTorch Architectures
The framework provides robust baselines (deepecg, resnet1d, transformer, etc.), but if you are researching novel neural network architectures, you will want to test your own model.

To integrate seamlessly with the biometric matching pipeline (which requires extracting embeddings for Gallery/Probe template matching), your custom model just needs to follow one simple rule: It must support returning feature embeddings when the classification head is disabled.

# Step 1: Write Your PyTorch Model
Your model should accept an include_top boolean argument. If False, it returns the high-dimensional feature vector. If True, it returns the Softmax logits.

In [ ]:
import torch
import torch.nn as nn

# 1. Define your custom architecture
class MyAwesomeECGNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=10, include_top=True):
        super(MyAwesomeECGNet, self).__init__()
        self.include_top = include_top

        # A simple Feature Extractor
        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Flattens to (Batch, 64, 1)
        )

        # The Biometric Embedding Layer (e.g., 64-dimensional embedding)
        self.embedding_size = 64

        # The Classification Head (Only used for training or Task 1/3)
        self.classifier = nn.Linear(self.embedding_size, num_classes)

    def forward(self, x):
        # x shape: (Batch, Channels, Time)
        x = self.features(x)
        embeddings = x.view(x.size(0), -1) # Flatten to (Batch, 64)

        # Framework Routing Logic:
        if not self.include_top:
            # For Tasks 2, 4, 6, 8 (Template Verification matching)
            return embeddings

        # For Training and Tasks 1, 3 (Identification)
        logits = self.classifier(embeddings)
        return logits

# Step 2: Register Your Model in the Framework
To make your model accessible via the CLI (main.py) or the standard run.py API, you just need to add it to the model dictionary.


1.   Open models.py in your repository.
2.   Paste your MyAwesomeECGNet class into the file.
3.   Open run.py (or wherever your _initialize_model helper function lives) and add your string tag to the routing dictionary.

Example of the modification inside your framework code:

In [ ]:
# Inside your framework's initialization logic (e.g., inside run.py)
from models import DeepECG, ResNet1D, MyAwesomeECGNet # <-- Import it

def get_model(model_name, num_classes, include_top=True):
    model_name = model_name.lower()

    if model_name == 'deepecg':
        return DeepECG(num_classes=num_classes, include_top=include_top)
    elif model_name == 'resnet1d':
        return ResNet1D(num_classes=num_classes, include_top=include_top)
    elif model_name == 'my_awesome_net': # <-- Register your custom tag!
        return MyAwesomeECGNet(num_classes=num_classes, include_top=include_top)
    else:
        raise ValueError(f"Model {model_name} not found.")

# Step 3: Run Your Benchmarks
Now, your custom model is fully integrated. You can benchmark it using the CLI just like the default models!

In [ ]:
# Run the Ultimate Cross-Session Verification Task with your new model!
!python main.py \
  --dataset heartprint \
  --task 8 \
  --data_split_mode cross-session \
  --train_sessions session1 \
  --probe_sessions session2 \
  --model my_awesome_net \
  --epochs 100 \
  --save_results